In [21]:
import numpy as np
import pandas as pd
from scipy.stats import linregress, pearsonr
from scipy.special import comb

from armored.models import *
from armored.preprocessing import *

import itertools

from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn import tree

In [2]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

# concatenate all independent variables
independent_variables = np.concatenate((np.array(species), np.array(controls)))

In [3]:
# predictions for all possible conditions
pred_df = pd.read_csv("space/space.csv")

In [4]:
# simulated data
E = np.array(pred_df.iloc[pred_df.Time.values==0]['Experiments'].values)
X = np.array(pred_df.iloc[pred_df.Time.values==0][species+controls].values) 
y = pred_df.iloc[pred_df.Time.values==3]['Butyrate'].values

# make X binary
X = np.array(X>0, int)

In [5]:
def find_rows_with_positive_values(X, columns):
    """
    Returns the indices of rows where all specified columns have values > 0.

    Parameters:
    X (numpy.ndarray): The input n x m matrix.
    columns (list): List of column indices to check.

    Returns:
    numpy.ndarray: Indices of rows where all specified columns are > 0.
    """
    # Create a boolean mask for the given columns
    mask = np.all(X[:, columns] > 0, axis=1)
    
    # Find and return the indices where the mask is True
    return np.where(mask)[0]

In [22]:
# save best motifs in a dictionary
motif_dict = {}

# Value of k
for k in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:

    # Loop over all n choose k subsets
    motifs = []
    min_vals = []
    total = comb(len(independent_variables), k) 
    for motif in tqdm(itertools.combinations(range(X.shape[-1]), k), total=total):
        
        # save motif name
        motifs.append(motif)
        
        # index of all samples that satisfy motif
        motif_idx = find_rows_with_positive_values(X, motif)    
        
        # save minimum predicted value
        min_vals.append(np.min(y[motif_idx]))
        
    # save best motif
    motif_observed = independent_variables[list(motifs[np.argmax(min_vals)])]
    
    # convert motif to hashable string
    motif_name = ""
    for m in motif_observed:
        motif_name += m.split('abs')[0] + '-'
    motif_name = motif_name[:-1]
    
    # save to dict
    motif_dict[motif_name] = np.max(min_vals)

100%|████████████████████████████████████████| 21/21.0 [00:00<00:00, 179.83it/s]
100%|█████████████████████████████████| 116280/116280.0 [28:33<00:00, 67.87it/s]
100%|█████████████████████████████████| 203490/203490.0 [58:18<00:00, 58.16it/s]
100%|███████████████████████████████| 293930/293930.0 [1:35:29<00:00, 51.31it/s]
100%|███████████████████████████████| 352716/352716.0 [2:12:56<00:00, 44.22it/s]


In [23]:
motif_dict

{'AC': 0.0,
 'AC-BA': 0.0,
 'AC-BA-BH': 0.0,
 'AC-BU-RI-Inulin': 0.5407897769780552,
 'AC-BU-CC-RI-Inulin': 1.907913965583253,
 'AC-BA-BU-CC-ER-Inulin': 3.3267112651952835,
 'AC-BL-BU-ER-RI-Starch-Xylan': 4.646532219099671,
 'AC-BA-BL-BU-ER-RI-Starch-Xylan': 5.915242032057876,
 'AC-BA-BL-BU-CC-ER-RI-Starch-Xylan': 6.809767073939193,
 'AC-BA-BL-BU-CC-ER-RI-Inulin-Starch-Xylan': 7.4592594627901265}

In [28]:
df = pd.DataFrame()
df['motifs'] = motif_dict.keys()
df['butyrate'] = motif_dict.values()

In [29]:
df

,motifs,butyrate
0,AC,0.000000
1,AC-BA,0.000000
2,AC-BA-BH,0.000000
3,AC-BU-RI-Inulin,0.540790
4,AC-BU-CC-RI-Inulin,1.907914
5,AC-BA-BU-CC-ER-Inulin,3.326711
6,AC-BL-BU-ER-RI-Starch-Xylan,4.646532
7,AC-BA-BL-BU-ER-RI-Starch-Xylan,5.915242
8,AC-BA-BL-BU-CC-ER-RI-Starch-Xylan,6.809767
9,AC-BA-BL-BU-CC-ER-RI-Inulin-Starch-Xylan,7.459259


In [30]:
df.to_csv("insights/maxi_min_motifs.csv", index=False)